### Incorporation of an external AI into Jupyter Notebook for Python
A free API of Gemini is used in this notebook. 

1. Create a text file named `.env` in the same folder as the notebook and enter the API key (never make this file public).
   GEMINI_API_KEY=A**** (your AIP key)
2. If you are managing the project with GitHub, add '.env' to your .gitignore file to prevent .env file from being uploaded to the public repository.

In [ ]:
# Integrating AI
!pip install google-genai

In [ ]:
# AIP environment
!pip install python-dotenv

In [ ]:
# List of available AI models
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

# List of available AI models
for model in client.models.list():
    print(model.name)

### Exercise of finding reduced echelon forms

Execute the next cell to create sliders. 

Choose the size $(m,n)$ of a matrix and click on "Create & Solve" buttun at the bottom of he next cell. Then the AI generates a $m{\times}n$ matrix whose row reduced echelon form should be computed, and the solution, which is verified by Python.   

In [ ]:
from google import genai
from IPython.display import Markdown, clear_output, display
import ipywidgets as widgets
import sympy as sp

# 1. Initialize the client
# client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# 2. Cretating sliders and an execute buttun
slider_m = widgets.IntSlider(
    value=3, min=2, max=5, step=1, description="Rows (m):"
)
slider_n = widgets.IntSlider(
    value=4, min=2, max=6, step=1, description="Cols (n):"
)
btn_generate = widgets.Button(
    description="Generate & Solve",
    button_style="primary",
    icon="play",
)
out = widgets.Output()


def on_button_clicked(b):
    with out:
        clear_output()
        # Getting slider values
        m = slider_m.value
        n = slider_n.value

        display(Markdown(f"*Generating a {m}×{n} problem and querying AI...*"))

        # 1. Python (SymPy) restricts the elements between -3 and 5
        A_sym = sp.randMatrix(m, n, min=-3, max=5)
        A_latex = sp.latex(A_sym)

        # 2. SymPy computes the correct answer
        true_rref, pivots = A_sym.rref()

        # 3. Asking AI to find the RREF
        prompt = f"""
Given the following {m} by {n} matrix A:
$${A_latex}$$

Find its reduced row echelon form (RREF) step by step.

Requirements:
1. For every elementary row operation, state the specific operation explicitly (e.g., $R_2 \\to R_2 - 2R_1$).
2. Immediately after each row operation, display the full intermediate matrix using LaTeX bmatrix format ($$\\begin{{bmatrix}} ... \\end{{bmatrix}}$$). Do not omit any rows.
3. End with the final reduced row echelon form matrix clearly labeled as "Final RREF".
"""

        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite", contents=prompt
            )
            ai_text = response.text
        except Exception as e:
            ai_text = f"API Error: {e}"

        clear_output()

        # 4. Showing the results on the Notebook
        display(
            Markdown(
                f"### An original matrix $A$ ({m}×{n})\n"
                f"$${A_latex}$$\n\n"
                f"### Step-by-step Solution generated by AI\n"
                f"{ai_text}\n\n"
                f"---\n"
                f"### Verification results (Python / SymPy)\n"
                f"**Ground Truth RREF:**\n"
                f"$${sp.latex(true_rref)}$$"
            )
        )


btn_generate.on_click(on_button_clicked)

# Setting the layout of UI
ui = widgets.VBox([widgets.HBox([slider_m, slider_n, btn_generate]), out])
display(ui)

### Exercise of finding reduced echelon forms

Execute the next cell to create sliders. 

Choose the size $n$ of a square matrix and click on "Create & Solve" buttun at the bottom of he next cell. Then the AI generates a $n{\times}n$ matrix whose determinant should be computed, and the solution, which is verified by Python.  

In [ ]:
from google import genai
from IPython.display import Markdown, clear_output, display
import ipywidgets as widgets
import sympy as sp

# 1. Initializing the client
# client = genai.Client()

# 2. Creating the slider and the execute button
slider_n = widgets.IntSlider(
    value=3, min=2, max=5, step=1, description="Size (n):"
)
btn_det = widgets.Button(
    description="Generate & Solve",
    button_style="primary",
    icon="calculator",
)
out_det = widgets.Output()


def on_det_button_clicked(b):
    with out_det:
        clear_output()
        n = slider_n.value
        display(
            Markdown(
                f"*Generating an invertible {n}×{n} matrix and querying AI...*"
            )
        )

        # Python (SymPy) restricts the elements between -3 and 5
        while True:
            A_sym = sp.randMatrix(n, n, min=-3, max=5)
            det_val = A_sym.det()
            if det_val != 0:
                break

        A_latex = sp.latex(A_sym)

        # 2. SymPy computes the correct answer
        inv_sym = A_sym.inv()

        # 3. Asking the AI to do 
        prompt = f"""
Given the following {n} by {n} integer matrix A:
$${A_latex}$$

Perform the following tasks step by step:

1. **Determinant Calculation:**
   - Compute the determinant of A using elementary row or column operations (e.g., transforming to an upper triangular matrix or simplifying before Laplace expansion).
   - Display each intermediate matrix explicitly using the LaTeX bmatrix format ($$\\begin{{bmatrix}} ... \\end{{bmatrix}}$$).
   - State each elementary operation performed.
   - Conclude clearly with "Determinant: $\\det(A) = \\dots$".

2. **Inverse Matrix Calculation:**
   - Since $\\det(A) \\neq 0$, compute the inverse matrix $A^{{-1}}$.
   - Explain the calculation steps using the elementary row or column operations on $[A \\mid I]$.
   - Display key intermediate steps and state clearly "Inverse Matrix: $A^{{-1}} = \\dots$" using LaTeX bmatrix. Use reduced fractions.
"""

        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite", contents=prompt
            )
            ai_text = response.text
        except Exception as e:
            ai_text = f"API Error: {e}"

        clear_output()

        # 4. Showing the results on the Notebook
        display(
            Markdown(
                f"### 1. Given Matrix $A$ ({n}×{n})\n"
                f"$${A_latex}$$\n\n"
                f"### 2. Step-by-Step Solution generated by AI\n"
                f"{ai_text}\n\n"
                f"---\n"
                f"### 3. Ground Truth Verification (Python / SymPy)\n"
                f"- **Strict Determinant:** $\\det(A) = {sp.latex(det_val)}$\n"
                f"- **Strict Inverse Matrix $A^{{-1}}$:**\n"
                f"$${sp.latex(inv_sym)}$$"
            )
        )


btn_det.on_click(on_det_button_clicked)

# The layout of the UI
ui_det = widgets.VBox([widgets.HBox([slider_n, btn_det]), out_det])
display(ui_det)